# train.ipynb — Entraînement du modèle
Adam + ReduceLROnPlateau + Early Stopping + sauvegarde du meilleur checkpoint.

## Configuration

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, '.')  # src/ est dans le path courant

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Imports locaux ────────────────────────────────────────────
%run dataset.ipynb   # charge get_dataloaders
%run model.ipynb     # charge get_model

# ── Hyperparamètres ───────────────────────────────────────────
DATA_DIR   = '../data/chest_xray'
ARCH       = 'baseline'     # 'baseline' | 'resnet18' | 'densenet121' | 'efficientnet_b0'
EPOCHS     = 30
BATCH_SIZE = 32
LR         = 1e-3
DROPOUT    = 0.5
PATIENCE   = 5
WORKERS    = 0              # 0 sous Windows
CKPT_DIR   = Path('../outputs/checkpoints')
FIG_DIR    = Path('../outputs/figures')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Appareil : {DEVICE.upper()}')

## Données + Modèle

In [ ]:
train_loader, val_loader, _ = get_dataloaders(DATA_DIR, BATCH_SIZE, WORKERS)

extra = {'dropout1': DROPOUT} if ARCH == 'baseline' else {}
model = get_model(ARCH, **extra).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modele : {ARCH}  |  Params : {n_params:,}')

pos_weight = torch.tensor([0.33]).to(DEVICE)   # gestion desequilibre
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = Adam(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=LR, weight_decay=1e-4)
try:
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
except TypeError:
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

## Boucle d'entraînement

In [ ]:
def accuracy(logits, labels):
    preds = (torch.sigmoid(logits) >= 0.5).long().squeeze(1)
    return (preds == labels).float().mean().item()

train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_loss  = float('inf')
no_improve     = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tl, ta = 0.0, 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.float().to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item(); ta += accuracy(logits.unsqueeze(1), lbls.long())
    tl /= len(train_loader); ta /= len(train_loader)

    model.eval()
    vl, va = 0.0, 0.0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.float().to(DEVICE)
            logits = model(imgs).squeeze(1)
            vl += criterion(logits, lbls).item()
            va += accuracy(logits.unsqueeze(1), lbls.long())
    vl /= len(val_loader); va /= len(val_loader)

    train_losses.append(tl); val_losses.append(vl)
    train_accs.append(ta);   val_accs.append(va)
    print(f'Ep {epoch:02d}/{EPOCHS} | TrainL:{tl:.4f} A:{ta:.4f} | ValL:{vl:.4f} A:{va:.4f} | {time.time()-t0:.1f}s')

    scheduler.step(vl)

    if vl < best_val_loss:
        best_val_loss = vl; no_improve = 0
        torch.save({'epoch': epoch, 'architecture': ARCH,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': vl, 'val_acc': va},
                   CKPT_DIR / 'best_model.pt')
        print(f'  -> Meilleur checkpoint sauvegarde (val_loss={vl:.4f})')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stopping - epoque {epoch}')
        break

print(f'Entrainement termine. Meilleure val_loss = {best_val_loss:.4f}')

## Courbes d'entraînement

In [ ]:
ep = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, train_losses, label='Train', color='royalblue', lw=2)
axes[0].plot(ep, val_losses,   label='Val',   color='tomato',    lw=2, linestyle='--')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, train_accs, label='Train', color='royalblue', lw=2)
axes[1].plot(ep, val_accs,   label='Val',   color='tomato',    lw=2, linestyle='--')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle(f'Entrainement — {ARCH}', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'training_curves.png', dpi=150)
%matplotlib inline
plt.show()
print('Courbes sauvegardees -> outputs/figures/training_curves.png')